# Custom Image Testing 

In [9]:
import sys
sys.path.append('..')

import torch
import matplotlib.pyplot as plt
from PIL import Image
import os

from src.model import DeepCNN
from src.utils import load_custom_image, predict_custom_images, load_custom_test_folder

device = torch.device('cpu')
class_names = ['Paper', 'Rock', 'Scissors']

## Load Best Model

In [10]:
model = DeepCNN(num_classes=3, dropout_rate=0.5).to(device)
model.load_state_dict(torch.load('../models/deep_cnn.pth', map_location=device))
model.eval()

DeepCNN(
  (conv1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv3): Sequential(
    (0): Conv2d(64, 128, ke

## Test on Custom Images

In [11]:
custom_data_path = '../data/custom_test'

if os.path.exists(custom_data_path):
    custom_images, true_labels, image_paths = load_custom_test_folder(custom_data_path, class_names)
    predictions, probabilities = predict_custom_images(model, custom_images, device)
    
    print(f"Tested {len(predictions)} custom images")
    
    correct = sum([pred == label for pred, label in zip(predictions, true_labels)])
    accuracy = correct / len(predictions)
    print(f"Custom Test Accuracy: {accuracy:.4f}")

Tested 21 custom images
Custom Test Accuracy: 0.3810


## Per-Class Performance on Custom Images

In [12]:
if os.path.exists(custom_data_path):
    from collections import Counter
    
    for i, class_name in enumerate(class_names):
        class_mask = [label == i for label in true_labels]
        class_correct = sum([pred == label for pred, label, mask in 
                           zip(predictions, true_labels, class_mask) if mask])
        class_total = sum(class_mask)
        
        if class_total > 0:
            class_acc = class_correct / class_total
            print(f"{class_name}: {class_acc:.4f} ({class_correct}/{class_total})")

Paper: 0.5714 (4/7)
Rock: 0.1429 (1/7)
Scissors: 0.4286 (3/7)
